# Data Validation & Quality

This notebook explores the **Validation & Quality** layer of `equity_lake`:

- **`ValidationPipeline`** — Orchestrates schema validation, profiling, and drift detection
- **`ValidationResult`** — Pydantic model capturing success, errors, warnings, and metrics
- **`PriceDataSchema`** / **`NewsDataSchema`** / **`MacroDataSchema`** — Pandera DataFrameModel schemas
- **`DataProfiler`** — Statistical profiling with optional whylogs backend
- **`DriftReport`** — Compare distributions between baseline and current data

**Prerequisites:** Run `equity ingest` to have price data available.

In [1]:
import sys
sys.path.insert(0, "../../src")

import pandas as pd
import numpy as np
from datetime import date, timedelta

try:
    from equity_lake.validation import (
        ValidationPipeline,
        ValidationResult,
        DataProfiler,
        DriftReport,
        PriceDataSchema,
        NewsDataSchema,
        MacroDataSchema,
        SCHEMA_REGISTRY,
    )
    print("Validation modules loaded")
except Exception as e:
    print(f"Import error: {e}")

try:
    from equity_lake.storage.duckdb import EquityDataDB
    print("Storage module loaded")
except Exception as e:
    print(f"Storage import error: {e}")

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    print("Plotly loaded")
except Exception as e:
    print(f"Plotly import error: {e}")

Validation modules loaded
Storage module loaded
Plotly loaded


/Users/minghao/Desktop/personal/equity_lake/.venv/lib/python3.12/site-packages/pandera/_pandas_deprecated.py:154: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


## 1. Pandera Schemas Overview

The `equity_lake` validation layer uses **Pandera** `DataFrameModel` schemas to enforce column types, value constraints, and cross-column checks at ingestion boundaries.

- **`PriceDataSchema`** — OHLCV with `price_consistency` (high>=low, high>=open/close) and `no_duplicates` (ticker+date)
- **`NewsDataSchema`** — News articles with `no_duplicate_urls` check, sentiment scores bounded [-1, 1]
- **`MacroDataSchema`** — Macro indicators with date, indicator name, value, and source

In [2]:
try:
    print("=== PriceDataSchema ===")
    print(f"Columns: {list(PriceDataSchema.to_schema().columns.keys())}")
    schema = PriceDataSchema.to_schema()
    for col_name, col in schema.columns.items():
        checks = [c for c in col.checks]
        dtype = col.dtype
        nullable = col.nullable
        print(f"  {col_name}: dtype={dtype}, nullable={nullable}, checks={len(checks)}")

    print("\nDataFrame-level checks:")
    for ck in schema.dataframe_checks:
        print(f"  - {ck.name}: {ck.error_message}")

    print("\n=== NewsDataSchema ===")
    print(f"Columns: {list(NewsDataSchema.to_schema().columns.keys())}")

    print("\n=== MacroDataSchema ===")
    print(f"Columns: {list(MacroDataSchema.to_schema().columns.keys())}")

    print(f"\nSCHEMA_REGISTRY keys: {list(SCHEMA_REGISTRY.keys())}")
except Exception as e:
    print(f"Error: {e}")

=== PriceDataSchema ===
Columns: ['ticker', 'date', 'open', 'high', 'low', 'close', 'volume', 'adj_close']
  ticker: dtype=str, nullable=False, checks=0
  date: dtype=datetime64[ns], nullable=False, checks=0
  open: dtype=float64, nullable=False, checks=1
  high: dtype=float64, nullable=False, checks=1
  low: dtype=float64, nullable=False, checks=1
  close: dtype=float64, nullable=False, checks=1
  volume: dtype=float64, nullable=False, checks=1
  adj_close: dtype=float64, nullable=False, checks=1

DataFrame-level checks:
Error: 'DataFrameSchema' object has no attribute 'dataframe_checks'


## 2. ValidationPipeline Demo

Create a `ValidationPipeline`, load price data from DuckDB, and validate against `PriceDataSchema`.

The pipeline performs three stages:
1. **Schema validation** via Pandera
2. **Data profiling** via `DataProfiler`
3. **Custom checks** (empty DataFrame, all-null columns, high-null columns)

In [3]:
try:
    pipeline = ValidationPipeline()
    print(f"Pipeline created (strict={pipeline.strict})")

    db = EquityDataDB()
    price_df = db.query("SELECT * FROM us_equity ORDER BY date DESC LIMIT 5000")
    db.close()

    if price_df.empty:
        print("No US equity data found. Generating synthetic data for demo.")
        np.random.seed(42)
        tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]
        dates = pd.date_range(end=date.today(), periods=60, freq="B")
        rows = []
        for t in tickers:
            base = np.random.uniform(100, 500)
            for d in dates:
                shock = np.random.normal(0, 0.02)
                close = round(base * (1 + shock), 2)
                high = round(close * (1 + abs(np.random.normal(0, 0.01))), 2)
                low = round(close * (1 - abs(np.random.normal(0, 0.01))), 2)
                opn = round(close * (1 + np.random.normal(0, 0.005)), 2)
                rows.append({"ticker": t, "date": d, "open": opn, "high": high,
                             "low": low, "close": close,
                             "volume": int(np.random.uniform(1e6, 5e7))})
        price_df = pd.DataFrame(rows)

    print(f"Loaded {len(price_df)} rows, columns: {list(price_df.columns)}")
    price_df.head()
except Exception as e:
    print(f"Error: {e}")

Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/krx_equity


Pipeline created (strict=False)


Loaded 5000 rows, columns: ['ticker', 'date', 'open', 'high', 'low', 'close', 'volume', 'adj_close', 'market']


## 3. Validation Results

Run `pipeline.validate()` against the `PriceDataSchema`. The `ValidationResult` Pydantic model captures:
- **success** — overall pass/fail
- **schema_valid** — Pandera check result
- **profile_valid** — profiling succeeded
- **drift_detected** — drift against baseline
- **errors** / **warnings** — human-readable messages
- **metrics** — quality and drift details

In [4]:
try:
    result = pipeline.validate(price_df, data_type="price", name="us_equity")

    print(f"success:       {result.success}")
    print(f"schema_valid:  {result.schema_valid}")
    print(f"profile_valid: {result.profile_valid}")
    print(f"drift_detected:{result.drift_detected}")
    print(f"errors:        {result.errors}")
    print(f"warnings:      {result.warnings}")
    print(f"timestamp:     {result.timestamp}")

    if result.metrics:
        print(f"\nMetrics keys: {list(result.metrics.keys())}")
        if "quality" in result.metrics:
            qm = result.metrics["quality"]
            for col, m in list(qm.items())[:5]:
                print(f"  {col}: completeness={m.get('completeness', 0):.2%}, "
                      f"nulls={m.get('null_count', 0)}, "
                      f"mean={m.get('mean', 'N/A')}")
except Exception as e:
    print(f"Error: {e}")

2026-06-09 09:26:22 [info     ] Created profile                name=us_equity rows=5000


success:       False
schema_valid:  False
profile_valid: True
drift_detected:False
errors:        ["Schema validation failed: non-nullable series 'open' contains null values:\n4811   NaN\n4812   NaN\n4813   NaN\n4814   NaN\n4815   NaN\n4816   NaN\n4817   NaN\n4818   NaN\n4819   NaN\n4820   NaN\n4821   NaN\n4822   NaN\n4823   NaN\n4824   NaN\n4825   NaN\n4826   NaN\n4827   NaN\n4828   NaN\n4829   NaN\n4830   NaN\n4831   NaN\n4832   NaN\n4833   NaN\n4834   NaN\n4835   NaN\n4836   NaN\n4837   NaN\n4838   NaN\n4839   NaN\n4840   NaN\nName: open, dtype: float64"]
warnings:      []
timestamp:     2026-06-09T01:26:22.640869+00:00

Metrics keys: ['quality']
  ticker: completeness=100.00%, nulls=0, mean=N/A
  date: completeness=100.00%, nulls=0, mean=1774484288640000.0
  open: completeness=99.40%, nulls=30, mean=275.0239429297342
  high: completeness=99.40%, nulls=30, mean=278.60032729565256
  low: completeness=99.40%, nulls=30, mean=271.43661137433116


## 4. Validation Pass/Fail Dashboard

Plotly indicator gauges showing the three validation dimensions: `schema_valid`, `profile_valid`, and `drift_detected`.

In [5]:
try:
    if "result" in dir():
        fig = make_subplots(
            rows=1, cols=3,
            specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
            subplot_titles=["Schema Valid", "Profile Valid", "Drift Detected"],
        )

        fig.add_trace(go.Indicator(
            mode="gauge+number",
            value=int(result.schema_valid),
            gauge={"axis": {"range": [0, 1]},
                   "bar": {"color": "#2CA02C" if result.schema_valid else "#D62728"},
                   "steps": [{"range": [0, 0.5], "color": "#FFD7D7"},
                             {"range": [0.5, 1], "color": "#D7FFD7"}]},
            number={"font": {"size": 40}},
            title={"text": "Schema"},
        ), row=1, col=1)

        fig.add_trace(go.Indicator(
            mode="gauge+number",
            value=int(result.profile_valid),
            gauge={"axis": {"range": [0, 1]},
                   "bar": {"color": "#2CA02C" if result.profile_valid else "#D62728"},
                   "steps": [{"range": [0, 0.5], "color": "#FFD7D7"},
                             {"range": [0.5, 1], "color": "#D7FFD7"}]},
            number={"font": {"size": 40}},
            title={"text": "Profile"},
        ), row=1, col=2)

        fig.add_trace(go.Indicator(
            mode="gauge+number",
            value=int(result.drift_detected),
            gauge={"axis": {"range": [0, 1]},
                   "bar": {"color": "#D62728" if result.drift_detected else "#2CA02C"},
                   "steps": [{"range": [0, 0.5], "color": "#D7FFD7"},
                             {"range": [0.5, 1], "color": "#FFD7D7"}]},
            number={"font": {"size": 40}},
            title={"text": "Drift"},
        ), row=1, col=3)

        fig.update_layout(title_text="Validation Dashboard", height=350)
        fig.show()
    else:
        print("No validation result. Run previous cells first.")
except Exception as e:
    print(f"Error: {e}")

## 5. Schema Compliance Heatmap

For each market, show which key columns pass or fail validation. Green (1) = passes, Red (0) = fails. We check for non-null values and valid ranges.

In [6]:
try:
    db = EquityDataDB()
    markets = {"us_equity": "us_equity", "cn_ashare": "cn_ashare",
              "hk_sg_equity": "hk_sg_equity", "jpx_equity": "jpx_equity",
              "krx_equity": "krx_equity"}

    check_cols = ["ticker", "date", "open", "high", "low", "close", "volume"]
    compliance = {}

    for label, view_name in markets.items():
        try:
            df = db.query(f"SELECT * FROM {view_name} LIMIT 1000")
            row = {}
            for col in check_cols:
                if col not in df.columns:
                    row[col] = 0
                elif df[col].isnull().all():
                    row[col] = 0
                elif col in ["open", "high", "low", "close"]:
                    row[col] = int((df[col] > 0).all())
                elif col == "volume":
                    row[col] = int((df[col] >= 0).all())
                else:
                    row[col] = int(df[col].notnull().all())
            compliance[label] = row
        except Exception:
            compliance[label] = {col: 0 for col in check_cols}

    db.close()

    heat_df = pd.DataFrame(compliance).T
    fig = px.imshow(
        heat_df.values,
        x=list(heat_df.columns),
        y=list(heat_df.index),
        color_continuous_scale=[["#D62728", "#D62728"], ["#2CA02C", "#2CA02C"]],
        range_color=[0, 1],
        title="Schema Compliance by Market",
        labels={"x": "Column", "y": "Market", "color": "Pass"},
        aspect="auto",
    )
    fig.update_xaxes(side="top")
    fig.update_layout(height=350)
    fig.show()
except Exception as e:
    print(f"Error: {e}")

Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/krx_equity


Query failed: Catalog Error: Table with name jpx_equity does not exist!
Did you mean "us_equity"?

LINE 1: SELECT * FROM jpx_equity LIMIT 1000
                      ^


Query failed: Catalog Error: Table with name krx_equity does not exist!
Did you mean "us_equity"?

LINE 1: SELECT * FROM krx_equity LIMIT 1000
                      ^


Error: 
    Invalid value of type 'builtins.list' received for the 'colorscale' property of imshow
        Received value: [['#D62728', '#D62728'], ['#2CA02C', '#2CA02C']]

    The 'colorscale' property is a colorscale and may be
    specified as:
      - A list of colors that will be spaced evenly to create the colorscale.
        Many predefined colorscale lists are included in the sequential, diverging,
        and cyclical modules in the plotly.colors package.
      - A list of 2-element lists where the first element is the
        normalized color level value (starting at 0 and ending at 1),
        and the second item is a valid color string.
        (e.g. [[0, 'green'], [0.5, 'red'], [1.0, 'rgb(0, 0, 255)']])
      - One of the following named colorscales:
            ['aggrnyl', 'agsunset', 'algae', 'amp', 'armyrose', 'balance',
             'blackbody', 'bluered', 'blues', 'blugrn', 'bluyl', 'brbg',
             'brwnyl', 'bugn', 'bupu', 'burg', 'burgyl', 'cividis', 'curl',
  

## 6. Drift Detection

Set a baseline profile from older data, then compare against newer data. The `DataProfiler.compare()` method checks if mean values in any column have drifted beyond a threshold (default 10%). Results are captured in a `DriftReport`.

In [7]:
try:
    db = EquityDataDB()
    full_df = db.query("SELECT * FROM us_equity ORDER BY date")
    db.close()

    if full_df.empty or len(full_df) < 100:
        print("Not enough data for drift demo. Using synthetic data.")
        np.random.seed(42)
        tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]
        dates = pd.date_range(end=date.today(), periods=120, freq="B")
        rows = []
        for t in tickers:
            base = np.random.uniform(100, 500)
            for d in dates:
                shock = np.random.normal(0, 0.02)
                close = round(base * (1 + shock), 2)
                high = round(close * (1 + abs(np.random.normal(0, 0.01))), 2)
                low = round(close * (1 - abs(np.random.normal(0, 0.01))), 2)
                opn = round(close * (1 + np.random.normal(0, 0.005)), 2)
                rows.append({"ticker": t, "date": d, "open": opn, "high": high,
                             "low": low, "close": close,
                             "volume": int(np.random.uniform(1e6, 5e7))})
        full_df = pd.DataFrame(rows)

    full_df["date"] = pd.to_datetime(full_df["date"])
    split_date = full_df["date"].quantile(0.5)
    baseline_df = full_df[full_df["date"] <= split_date]
    current_df = full_df[full_df["date"] > split_date]

    drift_pipeline = ValidationPipeline()
    drift_pipeline.set_baseline("price", baseline_df)

    drift_result = drift_pipeline.validate(current_df, data_type="price",
                                           check_drift=True, name="price")

    print(f"Drift detected: {drift_result.drift_detected}")
    if drift_result.metrics.get("drift"):
        drift_info = drift_result.metrics["drift"]
        print(f"Drift report: {drift_info}")
    else:
        print("No significant drift found between baseline and current data.")
except Exception as e:
    print(f"Error: {e}")

Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


Data directory not found: /Users/minghao/Desktop/personal/equity_lake/data/lake/krx_equity


2026-06-09 09:26:23 [info     ] Created profile                name=baseline_price rows=62950


2026-06-09 09:26:23 [info     ] Created profile                name=price rows=62932


Drift detected: True
Drift report: {'has_drift': True, 'columns': {'date': {'mean_current': 1701808001728850.5, 'mean_baseline': 1543895668308181.0, 'pct_change': 0.10228173876134498}, 'open': {'mean_current': 215.3064437098762, 'mean_baseline': 111.81149403985481, 'pct_change': 0.9256199513185199}, 'high': {'mean_current': 217.67330056205435, 'mean_baseline': 112.89883619346914, 'pct_change': 0.9280384802996411}, 'low': {'mean_current': 212.91312475795928, 'mean_baseline': 110.68627823576462, 'pct_change': 0.9235728958602154}, 'close': {'mean_current': 215.35251719178487, 'mean_baseline': 111.81545110096337, 'pct_change': 0.9259638544706408}, 'volume': {'mean_current': 25127668.5223769, 'mean_baseline': 31303162.389007147, 'pct_change': 0.19728019137130123}, 'adj_close': {'mean_current': 209.30014680859182, 'mean_baseline': 100.65194705704454, 'pct_change': 1.0794445902767371}}, 'profile_current': 'current', 'profile_baseline': 'baseline', 'timestamp': '2026-06-09T01:26:23.630937+00:0

### Drift Metrics Over Time

Simulate multiple time windows and track drift metrics across them, showing how distributional changes evolve.

In [8]:
try:
    if "full_df" in dir() and not full_df.empty:
        full_df["date"] = pd.to_datetime(full_df["date"])
        dates_sorted = full_df["date"].sort_values().unique()

        window_size = max(len(dates_sorted) // 6, 2)
        windows = []
        for i in range(0, len(dates_sorted) - window_size, window_size):
            win_dates = dates_sorted[i:i + window_size]
            windows.append(full_df[full_df["date"].isin(win_dates)])

        baseline_win = windows[0]
        dp = DataProfiler(storage_path="/tmp/nb10_profiles")
        baseline_profile = dp.profile(baseline_win, "drift_baseline")

        drift_records = []
        for idx, win in enumerate(windows[1:], start=1):
            try:
                cur_profile = dp.profile(win, f"drift_window_{idx}")
                report = dp.compare(cur_profile, baseline_profile)
                drift_records.append({
                    "window": idx,
                    "has_drift": int(report.has_drift),
                    "columns_affected": len(report.columns),
                })
            except Exception as inner_e:
                drift_records.append({"window": idx, "has_drift": 0, "columns_affected": 0})

        drift_df = pd.DataFrame(drift_records)

        fig = make_subplots(specs=[[{"secondary_y": True}]])
        fig.add_trace(go.Bar(
            x=drift_df["window"], y=drift_df["columns_affected"],
            name="Columns Affected", marker_color="#FF7F0E",
        ), secondary_y=False)
        fig.add_trace(go.Scatter(
            x=drift_df["window"], y=drift_df["has_drift"],
            mode="lines+markers", name="Drift Detected",
            line=dict(color="#D62728", width=2),
        ), secondary_y=True)
        fig.update_layout(
            title="Drift Metrics Across Time Windows",
            xaxis_title="Window Index",
            height=400,
        )
        fig.update_yaxes(title_text="Columns Affected", secondary_y=False)
        fig.update_yaxes(title_text="Drift (0/1)", secondary_y=True, range=[-0.1, 1.1])
        fig.show()
    else:
        print("No data available for drift-over-time analysis.")
except Exception as e:
    print(f"Error: {e}")

2026-06-09 09:26:23 [info     ] Created profile                name=drift_baseline rows=21000


2026-06-09 09:26:23 [info     ] Created profile                name=drift_window_1 rows=21000


2026-06-09 09:26:23 [info     ] Created profile                name=drift_window_2 rows=21000


2026-06-09 09:26:23 [info     ] Created profile                name=drift_window_3 rows=21000


2026-06-09 09:26:23 [info     ] Created profile                name=drift_window_4 rows=21000


## 7. validate_and_fix() Repair Workflow

The `validate_and_fix()` method automatically repairs common data quality issues:
- Removes duplicate `ticker + date` rows (keeping the last occurrence)
- Drops rows where `close <= 0`
- Fills null `volume` values with 0

After repair, it re-runs validation to confirm the fixes.

In [9]:
try:
    np.random.seed(99)
    messy_data = pd.DataFrame({
        "ticker": ["AAPL"] * 5 + ["MSFT"] * 3,
        "date": pd.to_datetime(["2024-01-02"] * 3 + ["2024-01-03"] * 2
                                + ["2024-01-02"] * 2 + ["2024-01-03"]),
        "open":  [180.0, 180.5, 181.0, 182.0, 183.0, 370.0, 371.0, 372.0],
        "high":  [182.0, 181.5, 183.0, 184.0, 185.0, 373.0, 374.0, 375.0],
        "low":   [179.0, 179.5, 180.0, 181.0, 182.0, 369.0, 370.0, 371.0],
        "close": [181.0, -1.0, 182.5, 183.5, 184.0, 372.0, 373.5, 374.0],
        "volume": [5e7, 4e7, np.nan, 6e7, 5.5e7, 3e7, np.nan, 3.5e7],
    })

    print(f"Original: {len(messy_data)} rows")
    dupes = messy_data.duplicated(subset=["ticker", "date"]).sum()
    neg_close = (messy_data["close"] <= 0).sum()
    print(f"  Duplicates (ticker+date): {dupes}")
    print(f"  Rows with close<=0: {neg_close}")
    null_vol = messy_data["volume"].isna().sum()
    print(f"  Null volumes: {null_vol}")

    fix_pipeline = ValidationPipeline()
    fixed_df, fix_result = fix_pipeline.validate_and_fix(messy_data, data_type="price")

    print(f"\nAfter fix: {len(fixed_df)} rows")
    print(f"  Validation success: {fix_result.success}")
    print(f"  Schema valid: {fix_result.schema_valid}")
    if fix_result.errors:
        print(f"  Errors: {fix_result.errors}")
    if fix_result.warnings:
        print(f"  Warnings: {fix_result.warnings}")

    fixed_df
except Exception as e:
    print(f"Error: {e}")

Original: 8 rows
  Duplicates (ticker+date): 4
  Rows with close<=0: 1
  Null volumes: 2
2026-06-09 09:26:23 [warning  ] Removed 4 duplicate rows      



After fix: 4 rows
  Validation success: True
  Schema valid: True


## 8. Data Profiling

The `DataProfiler` computes per-column statistics: null counts, cardinality, completeness, uniqueness, and (for numeric columns) distribution metrics (mean, std, min, max).

In [10]:
try:
    profiler = DataProfiler(storage_path="/tmp/nb10_profiles")
    sample_df = price_df.head(500) if len(price_df) >= 500 else price_df

    profile = profiler.profile(sample_df, "demo_profile")
    quality = profiler.get_quality_metrics(profile)

    rows = []
    for col_name, m in quality.items():
        rows.append({
            "column": col_name,
            "completeness": f"{m.get('completeness', 0):.2%}",
            "null_count": m.get("null_count", 0),
            "uniqueness": f"{m.get('uniqueness', 0):.2%}",
            "count": m.get("count", 0),
            "mean": f"{m['mean']:.4f}" if "mean" in m else "N/A",
            "std": f"{m['std']:.4f}" if "std" in m else "N/A",
            "min": f"{m['min']:.4f}" if "min" in m else "N/A",
            "max": f"{m['max']:.4f}" if "max" in m else "N/A",
        })

    profile_df = pd.DataFrame(rows)
    profile_df
except Exception as e:
    print(f"Error: {e}")

2026-06-09 09:26:23 [info     ] Created profile                name=demo_profile rows=500


### Profiling Visualization — Column Completeness

Bar chart showing data completeness (percentage of non-null values) per column.

In [11]:
try:
    if "quality" in dir():
        comp_data = pd.DataFrame([
            {"column": col, "completeness": m.get("completeness", 0) * 100}
            for col, m in quality.items()
        ])

        fig = px.bar(
            comp_data, x="column", y="completeness",
            title="Column Completeness (%)",
            labels={"completeness": "Completeness (%)"},
            color="completeness",
            color_continuous_scale=["#D62728", "#FF7F0E", "#2CA02C"],
            range_color=[0, 100],
        )
        fig.add_hline(y=95, line_dash="dash", line_color="gray",
                      annotation_text="95% threshold")
        fig.update_layout(height=400, showlegend=False)
        fig.show()
    else:
        print("No quality metrics available. Run previous cell first.")
except Exception as e:
    print(f"Error: {e}")

## 9. Next Steps

- **[02-data-ingestion](02-data-ingestion.ipynb)** — Ingest price data that feeds into validation
- **[03-storage-and-queries](03-storage-and-queries.ipynb)** — Query validated data from DuckDB